# 06 Final Model Evaluation

This notebook loads `models/best_model.joblib` and evaluates the saved model pipeline on the same leakage-safe held-out test split used during predictive modeling.

Outputs:

- `reports/final_model_metrics.csv`
- `reports/threshold_analysis.csv`
- Figures saved under `reports/figures/`

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score

# Make `src` importable when running from notebooks/ without `pip install -e .`.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "CardiacPatientData.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import FIGURES_DIR, MODELS_DIR, RAW_DATA_PATH, REPORTS_DIR
from src.evaluation import (
    bootstrap_ci,
    compute_metrics,
    plot_calibration_curve,
    plot_roc_pr,
)
from src.modeling import build_feature_matrix, load_model, make_leakage_safe_holdout

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODELS_DIR / "best_model.joblib"
FINAL_METRICS_PATH = REPORTS_DIR / "final_model_metrics.csv"
THRESHOLD_ANALYSIS_PATH = REPORTS_DIR / "threshold_analysis.csv"
sns.set_theme(style="whitegrid")


## Load data, recreate the held out test set, and score the saved model

The split logic mirrors `notebooks/05_predictive_modeling.ipynb`: repeated `ID` values are kept together with `StratifiedGroupKFold`; otherwise, a stratified row-wise split is used. Clinical features are deterministic row-level features and are added after splitting. `ID` is removed before prediction to avoid identifier leakage.

In [ ]:
model = load_model(MODEL_PATH)
df = pd.read_csv(RAW_DATA_PATH)

train_raw, test_raw, groups_train, split_strategy = make_leakage_safe_holdout(df)
X_test = build_feature_matrix(test_raw.drop(columns=["Outcome"]))
y_test = test_raw["Outcome"].astype(int)
test_groups = test_raw["ID"].to_numpy()

y_score = model.predict_proba(X_test)[:, 1]

print(f"Model loaded from: {MODEL_PATH}")
print(f"Split strategy: {split_strategy}")
print(f"Test rows: {len(test_raw):,}")
print(f"Test event rate: {y_test.mean():.3f}")
print(f"Score range: {y_score.min():.3f} to {y_score.max():.3f}")


## Threshold analysis

The table below evaluates clinically relevant tradeoffs across probability thresholds. Sensitivity is prioritized for a high-risk screening flag, while specificity and PPV quantify the false alarm burden.

In [ ]:
thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)
threshold_columns = [
    "threshold",
    "sensitivity",
    "specificity",
    "ppv",
    "npv",
    "f1",
    "flagged",
    "flagged_pct",
    "true_negative",
    "false_positive",
    "false_negative",
    "true_positive",
]
rows = []
for t in thresholds:
    m = compute_metrics(y_test, y_score, threshold=t)
    rows.append({key: m[key] for key in threshold_columns})
threshold_df = pd.DataFrame(rows)

# Clinical rule: keep at least 90% sensitivity when possible, then pick the
# highest such threshold to limit false alarms; otherwise pick best F1.
sensitive = threshold_df[threshold_df["sensitivity"] >= 0.90]
if len(sensitive):
    recommended_row = sensitive.sort_values(
        ["threshold", "specificity", "ppv"], ascending=False
    ).iloc[0]
    recommendation_rule = (
        "highest threshold with sensitivity >= 0.90 to limit false alarms"
    )
else:
    recommended_row = threshold_df.sort_values(
        ["f1", "sensitivity"], ascending=False
    ).iloc[0]
    recommendation_rule = "highest F1 because no threshold achieved sensitivity >= 0.90"

recommended_threshold = float(recommended_row["threshold"])
threshold_df["recommended"] = threshold_df["threshold"].eq(recommended_threshold)
threshold_df.to_csv(THRESHOLD_ANALYSIS_PATH, index=False)

print(f"Saved threshold analysis to: {THRESHOLD_ANALYSIS_PATH}")
print(f"Recommended threshold: {recommended_threshold:.2f} ({recommendation_rule})")
display(threshold_df)


## Discrimination, calibration, and patient-level bootstrap CIs

Headline metrics are reported with **95% confidence intervals from a
patient-level bootstrap** (`src.evaluation.bootstrap_ci`). Because a single
patient can contribute hundreds of rows, the bootstrap resamples unique patient
IDs (not rows) on the held-out test set only, so the intervals reflect the true
number of independent patients rather than the inflated row count.


In [ ]:
fig, _ = plot_roc_pr(y_test, y_score)
roc_pr_path = FIGURES_DIR / "final_model_roc_pr_curves.png"
fig.savefig(roc_pr_path, dpi=300, bbox_inches="tight")
plt.show()

# Patient-level bootstrap 95% CIs on the held-out test set. Sensitivity and
# specificity are evaluated at the recommended operating threshold; AUROC,
# AUPRC, and Brier score are threshold-independent.
N_BOOT = 1000
metric_fns = {
    "auroc": roc_auc_score,
    "auprc": average_precision_score,
    "sensitivity": lambda yt, yp: compute_metrics(yt, yp, recommended_threshold)[
        "sensitivity"
    ],
    "specificity": lambda yt, yp: compute_metrics(yt, yp, recommended_threshold)[
        "specificity"
    ],
    "brier_score": lambda yt, yp: compute_metrics(yt, yp)["brier_score"],
}

ci = {}
for name, fn in metric_fns.items():
    point, lower, upper = bootstrap_ci(
        y_test.to_numpy(), y_score, fn, groups=test_groups, n_boot=N_BOOT
    )
    ci[name] = {"point": point, "lower": lower, "upper": upper}


def fmt_ci(name):
    """Format as e.g. ``0.82 (95% CI: 0.74-0.89)``."""
    c = ci[name]
    return f"{c['point']:.2f} (95% CI: {c['lower']:.2f}–{c['upper']:.2f})"


n_patients = int(pd.Series(test_groups).nunique())
print(f"Saved ROC/PR curves to: {roc_pr_path}")
print(
    f"Held-out test set: {len(y_test):,} rows from {n_patients} unique patients "
    f"({N_BOOT} patient-level bootstrap resamples)."
)
print(f"Operating threshold for sensitivity/specificity: {recommended_threshold:.2f}")
for name in metric_fns:
    print(f"  {name:12s}: {fmt_ci(name)}")


## Calibration assessment

In [ ]:
brier = compute_metrics(y_test, y_score)["brier_score"]

ax = plot_calibration_curve(y_test, y_score, n_bins=10)
calibration_path = FIGURES_DIR / "final_model_calibration_curve.png"
ax.figure.savefig(calibration_path, dpi=300, bbox_inches="tight")
plt.show()

# Calibration intercept/slope via logistic recalibration on the logit of scores.
eps = np.finfo(float).eps
clipped = np.clip(y_score, eps, 1 - eps)
logit_scores = np.log(clipped / (1 - clipped))
calibration_model = sm.Logit(y_test, sm.add_constant(logit_scores)).fit(disp=False)
calibration_intercept = float(calibration_model.params.iloc[0])
calibration_slope = float(calibration_model.params.iloc[1])

print(f"Brier score: {brier:.4f}")
print(f"Calibration intercept: {calibration_intercept:.4f}")
print(f"Calibration slope: {calibration_slope:.4f}")
print(f"Saved calibration curve to: {calibration_path}")


## Confusion matrices at multiple thresholds

In [ ]:
thresholds_to_plot = sorted({0.10, 0.20, recommended_threshold, 0.50, 0.80})
n_cols = min(3, len(thresholds_to_plot))
n_rows = int(np.ceil(len(thresholds_to_plot) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4.5 * n_rows))
axes = np.atleast_1d(axes).ravel()

for ax, threshold in zip(axes, thresholds_to_plot):
    y_pred = (y_score >= threshold).astype(int)
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=["Predicted low", "Predicted high"],
        yticklabels=["Actual low", "Actual high"],
        ax=ax,
    )
    suffix = " (recommended)" if np.isclose(threshold, recommended_threshold) else ""
    ax.set_title(f"Threshold {threshold:.2f}{suffix}")
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("Actual class")

for ax in axes[len(thresholds_to_plot):]:
    ax.axis("off")

fig.suptitle("Held-out confusion matrices across thresholds", y=1.02, fontsize=16)
fig.tight_layout()
confusion_path = FIGURES_DIR / "final_model_confusion_matrices.png"
fig.savefig(confusion_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved confusion matrices to: {confusion_path}")


## Final metrics and clinical threshold recommendation

In [ ]:
def ci_columns(name):
    c = ci[name]
    return {
        name: c["point"],
        f"{name}_lower": c["lower"],
        f"{name}_upper": c["upper"],
    }


rec = recommended_row.to_dict()
final_metrics = {
    "model_path": str(MODEL_PATH),
    "split_strategy": split_strategy,
    "test_rows": len(test_raw),
    "test_patients": int(pd.Series(test_groups).nunique()),
    "test_event_rate": float(y_test.mean()),
    "threshold": recommended_threshold,
    "recommendation_rule": recommendation_rule,
    "n_boot": N_BOOT,
    "ci_method": "patient-level (group) bootstrap, 2.5/97.5 percentiles",
    **ci_columns("auroc"),
    **ci_columns("auprc"),
    **ci_columns("sensitivity"),
    **ci_columns("specificity"),
    **ci_columns("brier_score"),
    "calibration_intercept": calibration_intercept,
    "calibration_slope": calibration_slope,
    "recommended_ppv": rec["ppv"],
    "recommended_npv": rec["npv"],
    "recommended_flagged_high_risk": int(rec["flagged"]),
    "recommended_flagged_high_risk_pct": rec["flagged_pct"],
}
final_metrics_df = pd.DataFrame([final_metrics])
final_metrics_df.to_csv(FINAL_METRICS_PATH, index=False)

print(f"Saved final metrics to: {FINAL_METRICS_PATH}")
display(final_metrics_df.T.rename(columns={0: "value"}))

headline = ", ".join(f"{name} {fmt_ci(name)}" for name in metric_fns)
recommendation_text = (
    f"At the recommended threshold of {recommended_threshold:.2f} "
    f"('{recommendation_rule}'), the model flags {int(rec['flagged'])} of "
    f"{len(y_test)} held-out patients ({rec['flagged_pct']:.1%}) as high risk. "
    f"Headline held-out performance with patient-level 95% CIs: {headline}. "
    "All intervals come from resampling patients (not rows); validate with "
    "clinicians and recalibrate prospectively before any operational use."
)
print(recommendation_text)


## Conclusion

This notebook evaluated the final model on the held-out test set to measure how well it performs on unseen patient data. The evaluation focused on discrimination, calibration, and threshold selection. AUROC and AUPRC showed how well the model separated higher-risk and lower-risk patients, while the Brier score and calibration curve assessed how reliable the predicted probabilities were. The threshold analysis was especially important because a cardiac risk model should prioritize catching high-risk patients rather than relying on the default 0.50 cutoff.

This notebook provides a final check of model performance and shows how the model could be used as a screening tool. Before any clinical use, the model would still need external validation, subgroup testing, and clinical review.